### Material Weakness: training the price forecaster

This notebook does the same thing as `scripts/train_forecaster.py` (which `start.py` runs for you), one step per cell, so you can see each result. If PyTorch won't install on your computer, run it in Google Colab: upload the project folder and open this notebook from its `notebooks/` folder.

The model looks at the last 10 daily returns and predicts the next day's return. Run it from the `notebooks/` folder of the project.

In [ ]:
import sys, json
sys.path.append('..')  #so we can import from the ml/ folder

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from ml.data import list_tickers, load_prices, daily_returns
from ml.forecast import make_windows, WINDOW, WEIGHTS, META

#the network: 10 returns in -> 64 -> 32 -> 1 predicted return out
class Forecaster(nn.Module):
  def __init__(self, window = WINDOW):
    super().__init__()
    self.net = nn.Sequential(nn.Linear(window, 64),
                             nn.ReLU(),
                             nn.Linear(64, 32),
                             nn.ReLU(),
                             nn.Linear(32, 1))

  def forward(self, x):
    return self.net(x)

torch.manual_seed(22)

#### 1. Look at the data

In [ ]:
list_tickers()

In [ ]:
df = load_prices(list_tickers()[0])
df.head()

In [ ]:
plt.plot(df['date'], df['close'])
plt.title('Closing price')
plt.xlabel('Date')
plt.ylabel('Price ($)');

#### 2. Make windows and split by time

Each row of `X` is 10 days of returns; `y` is the next day. The first 80% of each ticker is training data and the last 20% is test data. We split by **time**, not randomly, so the model can't peek at the future.

In [ ]:
X_train, y_train, X_test, y_test = [], [], [], []
for ticker in list_tickers():
    r = daily_returns(load_prices(ticker)).values
    X, y = make_windows(r, WINDOW)
    cut = int(len(X) * 0.8)
    X_train.append(X[:cut]); y_train.append(y[:cut])
    X_test.append(X[cut:]); y_test.append(y[cut:])

X_train, y_train = np.concatenate(X_train), np.concatenate(y_train)
X_test, y_test = np.concatenate(X_test), np.concatenate(y_test)
X_train.shape, X_test.shape

#### 3. Scale with the training mean and std, then make tensors

In [ ]:
mean, std = float(X_train.mean()), float(X_train.std())
scale = lambda a: (a - mean) / std

Xtr = torch.tensor(scale(X_train), dtype = torch.float32)
ytr = torch.tensor(scale(y_train), dtype = torch.float32)
Xte = torch.tensor(scale(X_test), dtype = torch.float32)
yte = torch.tensor(scale(y_test), dtype = torch.float32)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size = 64, shuffle = True)

#### 4. Model, loss, optimizer

Regression: no activation at the end, `MSELoss`.

In [ ]:
model = Forecaster(WINDOW)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay = 1e-4)
model

#### 5. Train

In [ ]:
losses = []
for epoch in range(30):
    running_loss = 0.0
    for xb, yb in train_loader:
        #pass the batch through the model
        yhat = model(xb)
        #compare to the real next-day returns
        loss = loss_fn(yhat, yb)
        #clear gradients, backpropagate, step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
    losses.append(running_loss / len(Xtr))

plt.plot(losses, '--o')
plt.title('Training loss')
plt.xlabel('Epoch')
plt.ylabel('MSE (scaled)');

#### 6. Evaluate on the test set and compare to baselines

In [ ]:
model.eval()
with torch.no_grad():
    test_pred = model(Xte)

test_mse = loss_fn(test_pred, yte).item()
baseline_mse = loss_fn(torch.zeros_like(yte), yte).item()  #always predict the average return

zero = (0 - mean) / std  #a 0% return, after scaling
direction_acc = ((test_pred > zero) == (yte > zero)).float().mean().item()
up_share = (yte > zero).float().mean().item()

print(f'Test MSE: {test_mse:.4f} | baseline MSE: {baseline_mse:.4f}')
print(f'Direction accuracy: {direction_acc:.1%} | always guessing the more common direction: {max(up_share, 1 - up_share):.1%}')

**Interpretation:** if the test MSE is about the same as the baseline, the model isn't really predicting tomorrow. That's normal for stock returns, and it's the point the Spin Dashboard's footnote makes.

#### 7. Save for the app

In [ ]:
#saved as NumPy arrays so the website can run the model without PyTorch
np.savez(WEIGHTS, **{k: v.detach().numpy() for k, v in model.state_dict().items()})
META.write_text(json.dumps({
    'window': WINDOW, 'mean': mean, 'std': std,
    'test_mse': round(test_mse, 4), 'baseline_mse': round(baseline_mse, 4),
    'direction_accuracy': round(direction_acc, 4), 'up_day_share': round(up_share, 4),
    'tickers': list_tickers(), 'train_losses': [round(x, 4) for x in losses],
}, indent = 2))
print('saved')